# Framework Comparison — 3 Modes

Same model (MiniMax M2.5), same prompt, same tools, same trace.  
1. **DSPy ReAct (text-based)** — `react_native_fc.ReAct(use_native_function_calling=False)` + XMLAdapter
2. **DSPy ReAct (native FC)** — `react_native_fc.ReAct(use_native_function_calling=True)`
3. **PydanticAI Agent (native FC)** — different framework, native FC by default

In [15]:
import sys, os, tempfile, shutil
from pathlib import Path
from dotenv import load_dotenv

load_dotenv(Path("../.env"))
load_dotenv(Path("../../lerim-cloud/.env"))

sys.path.insert(0, str(Path("../src").resolve()))
sys.path.insert(0, str(Path("../../lerim-cloud").resolve()))

# ── FLAGS ──
RUN_DSPY_TEXT = True       # ReAct with use_native_function_calling=False (text-based)
RUN_DSPY_NATIVE_FC = True  # ReAct with use_native_function_calling=True (native FC)
RUN_PYDANTICAI = True      # PydanticAI Agent (native FC, different framework)

# ── MODEL: switch between "minimax" and "ollama" ──
USE_MODEL = "gpt-oss-20b"  # "minimax" or "gpt-oss-20b" or "gemma4-e2b"

MODEL_CONFIGS = {
    "minimax": {
        "dspy_provider": "minimax",
        "dspy_model": "MiniMax-M2.5",
        "pai_provider": "minimax",
        "pai_model": "MiniMax-M2.5",
    },
    "gpt-oss-20b": {
        "dspy_provider": "ollama",
        "dspy_model": "gpt-oss:20b",
        "pai_provider": "ollama",
        "pai_model": "gpt-oss:20b",
    },
    "gemma4-e2b": {
        "dspy_provider": "ollama",
        "dspy_model": "gemma4:E2b",
        "pai_provider": "ollama",
        "pai_model": "gemma4:E2b",
    },
}
MODEL_CFG = MODEL_CONFIGS[USE_MODEL]

# Trace
TRACE = Path("/Users/kargarisaac/codes/personal/lerim/lerim-cloud/evals/data/traces/claude_003.jsonl")
assert TRACE.exists(), f"Trace not found: {TRACE}"
print(f"Trace: {TRACE.name} ({sum(1 for _ in open(TRACE))} lines)")
print(f"Model: {USE_MODEL} ({MODEL_CFG['dspy_model']})")
print(f"Modes: text={RUN_DSPY_TEXT}, dspy_fc={RUN_DSPY_NATIVE_FC}, pydanticai={RUN_PYDANTICAI}")

Trace: claude_003.jsonl (157 lines)
Model: gpt-oss-20b (gpt-oss:20b)
Modes: text=True, dspy_fc=True, pydanticai=True


In [16]:
def make_temp_memory():
    """Create a fresh temp memory directory."""
    tmp = Path(tempfile.mkdtemp(prefix="fw_compare_"))
    mem = tmp / "memory"
    mem.mkdir(parents=True)
    (mem / "summaries").mkdir()
    (mem / "index.md").write_text("# Memory Index\n")
    return tmp, mem

def report(mem: Path):
    """Print what the agent wrote."""
    import frontmatter as fm_lib
    memories = [f for f in sorted(mem.glob("*.md")) if f.name != "index.md"]
    summaries = list(sorted((mem / "summaries").glob("*.md")))
    print(f"\nMemories: {len(memories)}")
    for m in memories:
        try:
            post = fm_lib.load(str(m))
            print(f"  [{post.get('type','')}] {m.name}")
            print(f"    {post.get('description','')[:100]}")
        except Exception as e:
            # Model wrote malformed YAML — show raw content instead of crashing
            print(f"  [YAML ERROR] {m.name} ({type(e).__name__})")
            print(f"    {m.read_text()[:200]}")
    print(f"Summaries: {len(summaries)}")
    for s in summaries:
        print(f"  {s.name}")
    print(f"Index:\n{(mem / 'index.md').read_text()[:500]}")

## 1. PydanticAI Agent (native FC, different framework)

In [ ]:
if RUN_PYDANTICAI:
    from lerim.agents.extract_pydanticai import build_model, build_extract_agent, ExtractDeps
    from lerim.agents.tools import MemoryTools

    tmp_pai, mem_pai = make_temp_memory()
    tools_pai = MemoryTools(memory_root=mem_pai, trace_path=TRACE)
    model = build_model(MODEL_CFG["pai_provider"], MODEL_CFG["pai_model"])
    agent = build_extract_agent(model)
    deps = ExtractDeps(tools=tools_pai)

    print(f"Running PydanticAI agent with {MODEL_CFG['pai_model']}...")
    result_pai = await agent.run("Extract memories from the session trace.", deps=deps)
    print(f"Summary: {result_pai.output.completion_summary}")
    report(mem_pai)

    # Conversation log
    print(f"\n{'=' * 70}")
    print(f"AGENT CONVERSATION LOG ({len(result_pai.all_messages())} messages)")
    print(f"{'=' * 70}")
    for i, msg in enumerate(result_pai.all_messages()):
        kind = msg.kind
        print(f"\n--- Message {i} [{kind}] ---")
        if kind == "request":
            for part in msg.parts:
                ptype = part.part_kind
                if ptype == "system-prompt":
                    print(f"  [system-prompt] ({len(part.content)} chars)")
                elif ptype == "user-prompt":
                    print(f"  [user-prompt] {part.content[:200]}")
                elif ptype == "tool-return":
                    print(f"  [tool-return] {part.tool_name} -> {str(part.content)[:200]}")
                else:
                    print(f"  [{ptype}] {str(part)[:200]}")
        elif kind == "response":
            for part in msg.parts:
                ptype = part.part_kind
                if ptype == "text":
                    print(f"  [text] {part.content[:300]}")
                elif ptype == "tool-call":
                    print(f"  [tool-call] {part.tool_name}({str(part.args)[:150]})")
                else:
                    print(f"  [{ptype}] {str(part)[:200]}")
else:
    print("SKIPPED (RUN_PYDANTICAI=False)")

Running PydanticAI agent with gpt-oss:20b...
